# 02 — Quantum-feature linear ridge: synthetic smoke check

This rerunnable prototype checks the package API, not molecular prediction quality or M4 completion.
It uses **synthetic, dimensionless inputs** and artificial debye-scale labels; no QM9 test sets,
IBM connections, credentials or hardware jobs are accessed.

The model fits a training-only scaler, encodes `gamma` times standardized inputs with fixed Qiskit gates,
and computes one local Z expectation per qubit. An expectation is the average result of a quantum
measurement, between -1 and 1. Classical `Ridge` learns the linear mapping to labels in debye;
all circuit parameters are data, not variational gate weights.

The four-input baseline below keeps the legacy one-column-per-qubit ZZ defaults with local
`StatevectorEstimator`. Two complete circuit passes avoid constant Z-only readout from a single
phase-encoding layer. Section 3 compares configurable encodings/qubits on the same 24-input arrays
using local matrix product state (MPS) simulation. Defaults are illustrative, not tuned; this does
not select a final architecture. See `docs/PLAN.md` §7.9 and `tests/test_quantum.py`.

In [ ]:
try:
    from qm9dipole.provenance import check_environment
except ModuleNotFoundError as e:
    raise RuntimeError(
        'Install the project requirements and select its environment as the notebook kernel; see README.md.'
    ) from e

print(check_environment())

## 1. Exercise the regressor on small synthetic arrays

Four columns correspond to four qubits. The quantum circuit is fixed; only the scaler and
ridge coefficients are fitted. Scaling is refitted on every call to `fit`, including each CV fold.
The validation rows below are synthetic development examples, not the reserved molecular test sets.

In [ ]:
import numpy as np
import pandas as pd

from qm9dipole.models.quantum import QuantumRidgeRegressor

rng = np.random.default_rng(2026)
X_train = rng.normal(size=(24, 4))
X_validation = rng.normal(size=(4, 4))
y_train = 2.0 + 0.5 * np.sin(X_train[:, 0]) - 0.2 * X_train[:, 1]
y_validation = 2.0 + 0.5 * np.sin(X_validation[:, 0]) - 0.2 * X_validation[:, 1]

model = QuantumRidgeRegressor(alpha=1.0, gamma=1.0, reps=2)
model.fit(X_train, y_train)
model.get_params()

## 2. Inspect features and predictions

The features are dimensionless expectations. Predictions are in the same units as the labels,
and negative magnitude predictions are clipped to zero by default. No accuracy claim is made from this table.

In [ ]:
phi = model.quantum_features(X_validation)
predictions = model.predict(X_validation)
assert phi.shape == (len(X_validation), model.pipeline_.named_steps["quantum"].n_qubits_)
assert predictions.shape == (len(X_validation),)
assert np.isfinite(phi).all() and np.isfinite(predictions).all()
assert np.all(np.abs(phi) <= 1.0 + 1e-12)

table = pd.DataFrame(
    phi, columns=model.pipeline_.named_steps['quantum'].get_feature_names_out()
)
table['synthetic_label_D'] = y_validation
table['prediction_D'] = predictions
table

In [ ]:
model.pipeline_.named_steps['quantum'].circuit_.draw(output='text')

## 3. Exploratory 24-input comparison: qubits and encodings are independent

Generate one shared synthetic training/validation dataset (40/8 rows, seed 2026) before the loop.
All nine configurations use **exactly the same X/y arrays**, with the unchanged `alpha=1`, `gamma=1`,
`reps=2`, `batch_size=128` and `clip_negative=True`. No tuning or final model selection is performed.
A training-label mean is the reference baseline; these artificial labels are not QM9 measurements.

The package defines ordered encoding-axis constants: ZZ has one phase slot, RY one angle slot,
and RY/RZ two ordered angle slots per qubit/upload. With `A` slots and `Q` qubits, capacity is `Q*A`.
Zero-based input `i` maps to `upload=i//capacity`, `qubit=(i%capacity)//A`, `rotation=i%A`;
unused slots are zero. `n_upload_layers_ = ceil(n_inputs/capacity)` counts uploads per complete pass,
and `reps` repeats all input passes. Thus 24 inputs fit 8 qubits without PCA: three uploads for ZZ/RY,
two for RY/RZ. The measured feature width is **Q**, not the 24 input columns.

ZZ uses built-in `zz_feature_map` blocks with `reps=1` and requires at least two complete passes.
RY and RY/RZ use library gates, a fixed `RX(pi/4)` mixer after each upload, and neighboring CZ gates
between steps; they allow `reps >= 1`. The mixer exposes phase inputs to Z readout. None of these
circuits has trainable gate weights. Changing the encoding/qubit count changes the model, not merely
a lossless input reshape.

MPS here uses Aer exact expectations (precision zero, truncation threshold zero), without finite shots
or noise. It can be efficient for this shallow nearest-neighbor structure, not arbitrary circuits.
The table shows synthetic validation MAE/RMSE in the artificial debye scale, logical depth/two-qubit
counts **before hardware transpilation**, and local elapsed time for fit, prediction and feature checks.
It does not measure QPU cost or demonstrate quantum advantage.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

rng_24 = np.random.default_rng(2026)
n_inputs = 24
X_train_24 = rng_24.normal(size=(40, n_inputs))
X_validation_24 = rng_24.normal(size=(8, n_inputs))
y_train_24 = (
    2.0 + 0.4 * np.sin(X_train_24[:, 0])
    + 0.2 * np.cos(X_train_24[:, 1]) - 0.1 * X_train_24[:, 2]
)
y_validation_24 = (
    2.0 + 0.4 * np.sin(X_validation_24[:, 0])
    + 0.2 * np.cos(X_validation_24[:, 1]) - 0.1 * X_validation_24[:, 2]
)
assert X_train_24.shape == (40, 24) and y_train_24.shape == (40,)
assert X_validation_24.shape == (8, 24) and y_validation_24.shape == (8,)
print(f"Shared input shapes: train={X_train_24.shape}, validation={X_validation_24.shape}")

mean_model_24 = DummyRegressor(strategy="mean").fit(X_train_24, y_train_24)
mean_predictions_24 = mean_model_24.predict(X_validation_24)
pd.DataFrame([{
    "baseline": "training-label mean",
    "MAE_D": mean_absolute_error(y_validation_24, mean_predictions_24),
    "RMSE_D": root_mean_squared_error(y_validation_24, mean_predictions_24),
}])

In [ ]:
from time import perf_counter

from qm9dipole.models.quantum import ENCODING_AXES

comparison_rows = []
for encoding in ("zz", "ry", "ry_rz"):
    for n_qubits in (8, 16, 24):
        candidate = QuantumRidgeRegressor(
            n_qubits=n_qubits,
            encoding=encoding,
            simulation_method="matrix_product_state",
        )
        started = perf_counter()
        candidate.fit(X_train_24, y_train_24)
        predictions_24 = candidate.predict(X_validation_24)
        features_24 = candidate.quantum_features(X_validation_24)
        elapsed_seconds = perf_counter() - started
        encoder = candidate.pipeline_.named_steps["quantum"]
        capacity = n_qubits * len(ENCODING_AXES[encoding])

        assert encoder.n_features_in_ == n_inputs
        assert encoder.n_features_out_ == n_qubits
        assert encoder.n_upload_layers_ == (n_inputs + capacity - 1) // capacity
        assert features_24.shape == (len(X_validation_24), n_qubits)
        assert predictions_24.shape == (len(X_validation_24),)
        assert np.isfinite(features_24).all() and np.isfinite(predictions_24).all()
        assert np.all(np.abs(features_24) <= 1.0 + 1e-12)

        circuit = encoder.circuit_
        comparison_rows.append({
            "encoding": encoding,
            "n_inputs": n_inputs,
            "n_qubits": n_qubits,
            "n_upload_layers": encoder.n_upload_layers_,
            "MAE_D": mean_absolute_error(y_validation_24, predictions_24),
            "RMSE_D": root_mean_squared_error(y_validation_24, predictions_24),
            "logical_depth": circuit.depth(),
            "two_qubit_gates": sum(item.operation.num_qubits == 2 for item in circuit.data),
            "elapsed_seconds": elapsed_seconds,
        })

comparison = pd.DataFrame(comparison_rows)
assert len(comparison) == 9
print("All 9 dimension checks passed: quantum features=(8, Q), predictions=(8,).")
comparison

## Before molecular benchmarking

- Supply a fixed-width descriptor table aligned by molecule ID and the corresponding debye labels.
- PCA is optional, not required to fit 24 inputs onto fewer qubits. If learned, place it upstream of
  this regressor in the same scikit-learn `Pipeline` so it is fitted on each training/CV fold;
  do not cross-validate globally fitted PCA scores.
- Use invariant molecular descriptors: encoding flattened XYZ does not create rotation, translation or atom-order invariance.
- `StandardScaler` is training-fitted, then `gamma` scales its scores. Do not divide arbitrary
  descriptors by 255; that scale is appropriate only for known 8-bit pixels.
- Compare matched classical and quantum models on identical training subsets/folds and tune only on training data.
- Freeze settings before the separate final test evaluation. The synthetic table does not choose a final model.

All encodings in this module use exact local simulation only. Finite shots/noise and a hardware
adapter are separate future work; any real IBM job requires approval first. MPS efficiency depends
on shallow nearest-neighbor structure and is not guaranteed for arbitrary circuits. This prototype
neither completes M4 nor commits a final hardware choice, and makes no quantum-advantage claim.